In [4]:
"""
EXPORT CONTROLS DATA GENERATION SCRIPT WITH DATES
Generates synthetic U.S. export compliance dataset with realistic dates
"""

import pandas as pd
import numpy as np
import random
from datetime import datetime, timedelta
import os

# Set random seed for reproducibility
np.random.seed(42)
random.seed(42)

print("="*70)
print("📦 EXPORT CONTROLS DATA GENERATOR (WITH DATES)")
print("="*70)
print("\nGenerating synthetic export compliance dataset with dates...")

def generate_baseline_data(n_rows):
    """Generate clean baseline dataset before injecting defects"""
    
    print(f"  ├─ Creating {n_rows:,} baseline records...")
    
    # Generate date range (last 12 months)
    end_date = datetime.now().replace(hour=0, minute=0, second=0, microsecond=0)
    start_date = end_date - timedelta(days=365)
    
    # Random shipment dates within the last year
    shipment_dates = []
    for _ in range(n_rows):
        random_days = np.random.randint(0, 365)
        shipment_date = start_date + timedelta(days=random_days)
        shipment_dates.append(shipment_date)
    
    # Clearance dates (shipment date + clearance hours)
    clearance_dates = []
    for i in range(n_rows):
        clearance_hours = np.random.randint(12, 72)
        clearance_date = shipment_dates[i] + timedelta(hours=clearance_hours)
        clearance_dates.append(clearance_date)
    
    data = {
        'shipment_id': [f"EXP-{100000 + i}" for i in range(n_rows)],
        'shipment_date': shipment_dates,
        'clearance_date': clearance_dates,
        
        'exporter_business_unit': np.random.choice(
            ['US-MedicalDevices', 'US-ConsumerElectronics', 'US-AerospaceParts', 
             'US-IndustrialEquipment', 'US-Pharmaceuticals'],
            n_rows, 
            p=[0.25, 0.30, 0.15, 0.20, 0.10]
        ),
        
        'consignee_name': np.random.choice([
            'GLOBAL LOGISTICS LTD', 'AL-KHOBAR DISTRIBUTORS', 
            'POLY TECHNOLOGIES', 'MED-SUPPLY-EU', 'ASIA TECH IMPORTS',
            'EURO MEDICAL SOLUTIONS', 'MIDDLE EAST TRADING CO',
            'NORTH STAR SUPPLY', 'PACIFIC TRADING GROUP', 'ATLANTIC DISTRIBUTORS'
        ], n_rows),
        
        'consignee_address': np.random.choice([
            '123 Corporate Blvd, Dublin, IE',
            '456 Industrial Zone, Riyadh, SA',
            '789 Haidian District, Beijing, CN',
            '321 Avenue de la Gare, Brussels, BE',
            '567 Business Park, Singapore, SG',
            '890 Technology Hub, Taipei, TW',
            '234 Innovation Drive, Bangalore, IN',
            '678 Commerce Street, Mexico City, MX'
        ], n_rows),
        
        'schedule_b_code': np.random.choice([
            '8471.30.0100', '3004.90.9203', '8542.31.0000', 
            '9018.90.7500', '8803.30.0030', '9022.14.0000',
            '8517.62.0010', '8708.29.5060', '9030.90.0000'
        ], n_rows),
        
        'eccn': np.random.choice(
            ['EAR99', '5A002', '3A001', '5A001', '9A004'],
            n_rows, 
            p=[0.45, 0.20, 0.15, 0.10, 0.10]
        ),
        
        'bom_origin_country': np.random.choice(
            ['US', 'IE', 'CN', 'KR', 'MX', 'DE', 'JP', 'TW', 'SG'],
            n_rows, 
            p=[0.30, 0.10, 0.15, 0.10, 0.10, 0.08, 0.07, 0.05, 0.05]
        ),
        
        'bom_tooling_origin': np.random.choice(
            ['US', 'DE', 'JP', 'TW', 'KR', 'IL'],
            n_rows, 
            p=[0.35, 0.20, 0.15, 0.12, 0.10, 0.08]
        ),
        
        'item_quantity': np.random.randint(1, 500, n_rows),
        
        'unit_value_usd': np.round(np.random.uniform(10.0, 5000.0, n_rows), 2),
        
        'license_type': np.random.choice(
            ['NLR', 'LICENSE', 'EXCEPTION'],
            n_rows, 
            p=[0.45, 0.35, 0.20]
        ),
        
        'license_obtained': np.random.choice(['Y', 'N'], n_rows, p=[0.85, 0.15]),
        
        'port_of_exit': np.random.choice([
            'CHICAGO O\'HARE', 'LOS ANGELES', 'NEWARK', 
            'MIAMI', 'HOUSTON', 'SEATTLE', 'NEW YORK'
        ], n_rows),
        
        'clearance_hours': np.random.randint(12, 72, n_rows),
        
        'pga_required': np.random.choice(['Y', 'N'], n_rows, p=[0.20, 0.80]),
        
        'pga_obtained': np.random.choice(['Y', 'N'], n_rows, p=[0.90, 0.10]),
        
        'trusted_trader_status': np.random.choice(['Y', 'N'], n_rows, p=[0.40, 0.60])
    }
    
    df = pd.DataFrame(data)
    
    # Calculate total value
    df['total_value_usd'] = df['item_quantity'] * df['unit_value_usd']
    
    # Parse consignee country from address (last part after comma)
    df['consignee_country'] = df['consignee_address'].str.split(',').str[-1].str.strip()
    
    # Add month and year columns for easy grouping
    df['shipment_month'] = df['shipment_date'].dt.month
    df['shipment_year'] = df['shipment_date'].dt.year
    df['shipment_quarter'] = df['shipment_date'].dt.quarter
    
    # Day of week (Monday=0, Sunday=6)
    df['shipment_day_of_week'] = df['shipment_date'].dt.dayofweek
    
    # Calculate actual clearance date based on hours
    df['clearance_date'] = df['shipment_date'] + pd.to_timedelta(df['clearance_hours'], unit='h')
    
    return df

def inject_compliance_defects(df):
    """Systematically inject hidden compliance errors"""
    
    print("  ├─ Injecting compliance defects...")
    df_defect = df.copy()
    
    # Track defects for reporting
    defects_injected = {}
    
    # ----------------------------------------
    # DEFECT 1: Address String Blindspot (2% probability)
    # Hidden sanctioned locations in address field
    # ----------------------------------------
    san_defect_mask = np.random.random(len(df_defect)) < 0.02
    sanctioned_locations = [
        ', PYONGYANG, NORTH KOREA',
        ', TEHRAN, IRAN',
        ', DAMASCUS, SYRIA',
        ', SIMFEROPOL, CRIMEA'
    ]
    df_defect.loc[san_defect_mask, 'consignee_address'] = (
        df_defect.loc[san_defect_mask, 'consignee_address'] + 
        np.random.choice(sanctioned_locations, size=san_defect_mask.sum())
    )
    df_defect.loc[san_defect_mask, 'consignee_country'] = np.random.choice(
        ['NORTH KOREA', 'IRAN', 'SYRIA', 'CRIMEA'],
        size=san_defect_mask.sum()
    )
    defects_injected['Address Blindspot'] = san_defect_mask.sum()
    
    # ----------------------------------------
    # DEFECT 2: Foreign Direct Product Rule Failure (1.5% probability)
    # Items made abroad with US tooling misclassified as EAR99
    # ----------------------------------------
    fdpr_defect_mask = (
        (df_defect['bom_origin_country'] != 'US') & 
        (df_defect['bom_tooling_origin'] == 'US') &
        (df_defect['eccn'].isin(['5A002', '3A001', '5A001', '9A004'])) &
        (np.random.random(len(df_defect)) < 0.015)
    )
    # Misclassify controlled items as EAR99
    df_defect.loc[fdpr_defect_mask, 'eccn'] = 'EAR99'
    defects_injected['FDPR Misclassification'] = fdpr_defect_mask.sum()
    
    # ----------------------------------------
    # DEFECT 3: PGA Approval Missing (1% probability)
    # Medical/pharma shipments without required PGA token
    # ----------------------------------------
    pga_missing_mask = (
        (df_defect['pga_required'] == 'Y') & 
        (np.random.random(len(df_defect)) < 0.01)
    )
    df_defect.loc[pga_missing_mask, 'pga_obtained'] = 'N'
    defects_injected['PGA Missing'] = pga_missing_mask.sum()
    
    # ----------------------------------------
    # DEFECT 4: License Mismatch (1% probability)
    # Declared license but no license obtained
    # ----------------------------------------
    license_mismatch_mask = (
        (df_defect['license_type'].isin(['LICENSE', 'EXCEPTION'])) &
        (np.random.random(len(df_defect)) < 0.01)
    )
    df_defect.loc[license_mismatch_mask, 'license_obtained'] = 'N'
    defects_injected['License Mismatch'] = license_mismatch_mask.sum()
    
    # ----------------------------------------
    # DEFECT 5: Export to Restricted Entity (0.5% probability)
    # Consignee name matches Entity List entity
    # ----------------------------------------
    restricted_entities = [
        'ZTE CORPORATION', 'HUAWEI TECHNOLOGIES', 
        'SICHUAN AEROSPACE', 'NORTH KOREA TRADING',
        'SMIC INTERNATIONAL', 'CHINA ELECTRONICS'
    ]
    entity_defect_mask = np.random.random(len(df_defect)) < 0.005
    df_defect.loc[entity_defect_mask, 'consignee_name'] = np.random.choice(
        restricted_entities, size=entity_defect_mask.sum()
    )
    defects_injected['Restricted Entity'] = entity_defect_mask.sum()
    
    # ----------------------------------------
    # DEFECT 6: Value Threshold Violations (systematic)
    # High-value shipments under NLR - these will be caught by analysis
    # ----------------------------------------
    eei_violations = ((df_defect['license_type'] == 'NLR') & (df_defect['total_value_usd'] > 2500)).sum()
    defects_injected['EEI Threshold Violations'] = eei_violations
    
    # ----------------------------------------
    # DEFECT 7: Delayed Shipments (systematic)
    # Some ports have longer clearance times
    # ----------------------------------------
    # Inject delays in specific ports
    delayed_ports = ['NEWARK', 'MIAMI']
    delay_mask = df_defect['port_of_exit'].isin(delayed_ports) & (np.random.random(len(df_defect)) < 0.3)
    df_defect.loc[delay_mask, 'clearance_hours'] += np.random.randint(24, 96)
    # Recalculate clearance date
    df_defect.loc[delay_mask, 'clearance_date'] = df_defect.loc[delay_mask, 'shipment_date'] + pd.to_timedelta(df_defect.loc[delay_mask, 'clearance_hours'], unit='h')
    defects_injected['Port Delays'] = delay_mask.sum()
    
    # Add a few specific hardcoded examples for testing/validation
    df_defect.loc[12, 'consignee_name'] = 'PEACEFUL MEDICAL SUPPLIES'
    df_defect.loc[12, 'consignee_address'] = 'KORDAY ROAD, PYONGYANG, NORTH KOREA'
    df_defect.loc[12, 'consignee_country'] = 'NORTH KOREA'
    
    df_defect.loc[45, 'eccn'] = 'EAR99'
    df_defect.loc[45, 'bom_origin_country'] = 'CN'
    df_defect.loc[45, 'bom_tooling_origin'] = 'US'
    
    df_defect.loc[88, 'unit_value_usd'] = 3500.00
    df_defect.loc[88, 'item_quantity'] = 1
    df_defect.loc[88, 'total_value_usd'] = 3500.00
    df_defect.loc[88, 'license_type'] = 'NLR'
    
    defects_injected['Hardcoded Examples'] = 3
    
    return df_defect, defects_injected

def generate_dataset():
    """Main function to generate and save the dataset"""
    
    # Generate data
    n_rows = 5000
    df_clean = generate_baseline_data(n_rows)
    df_final, defects = inject_compliance_defects(df_clean)
    
    # Save dataset
    filename = 'us_export_compliance_data.csv'
    df_final.to_csv(filename, index=False)
    
    # Generate summary statistics
    print("  ├─ Calculating summary statistics...")
    
    summary = {
        'Total Rows': len(df_final),
        'Total Columns': len(df_final.columns),
        'Unique ECCN Codes': df_final['eccn'].nunique(),
        'Unique Countries': df_final['consignee_country'].nunique(),
        'Unique Ports': df_final['port_of_exit'].nunique(),
        'Date Range': f"{df_final['shipment_date'].min().strftime('%Y-%m-%d')} to {df_final['shipment_date'].max().strftime('%Y-%m-%d')}",
        'NLR Shipments': len(df_final[df_final['license_type'] == 'NLR']),
        'License Shipments': len(df_final[df_final['license_type'] == 'LICENSE']),
        'Exception Shipments': len(df_final[df_final['license_type'] == 'EXCEPTION']),
        'Average Shipment Value': f"${df_final['total_value_usd'].mean():,.2f}",
        'Total Shipment Value': f"${df_final['total_value_usd'].sum():,.2f}",
        'Average Clearance Hours': f"{df_final['clearance_hours'].mean():.1f} hrs",
        'Address Red Flags': defects['Address Blindspot'],
        'FDPR Defects': defects['FDPR Misclassification'],
        'PGA Defects': defects['PGA Missing'],
        'License Mismatches': defects['License Mismatch'],
        'Restricted Entities': defects['Restricted Entity'],
        'EEI Violations': defects['EEI Threshold Violations'],
        'Port Delays': defects['Port Delays']
    }
    
    # Print results
    print("\n" + "="*70)
    print("✅ DATASET GENERATED SUCCESSFULLY!")
    print("="*70)
    
    print("\n📊 DATASET SUMMARY:")
    print(f"  ├─ File: {filename}")
    print(f"  ├─ Location: {os.getcwd()}\\{filename}")
    print(f"  ├─ Total Shipments: {summary['Total Rows']:,}")
    print(f"  ├─ Total Columns: {summary['Total Columns']}")
    print(f"  ├─ Date Range: {summary['Date Range']}")
    print(f"  ├─ Average Shipment Value: {summary['Average Shipment Value']}")
    print(f"  ├─ Total Value: {summary['Total Shipment Value']}")
    print(f"  ├─ Average Clearance: {summary['Average Clearance Hours']}")
    
    print("\n🔴 DEFECTS INJECTED:")
    print(f"  ├─ Address Blindspots: {summary['Address Red Flags']}")
    print(f"  ├─ FDPR Misclassifications: {summary['FDPR Defects']}")
    print(f"  ├─ PGA Missing Approvals: {summary['PGA Defects']}")
    print(f"  ├─ License Mismatches: {summary['License Mismatches']}")
    print(f"  ├─ Restricted Entities: {summary['Restricted Entities']}")
    print(f"  ├─ EEI Threshold Violations: {summary['EEI Violations']}")
    print(f"  ├─ Port Delays Injected: {summary['Port Delays']}")
    print(f"  └─ Hardcoded Examples: 3 (rows 12, 45, 88)")
    
    print("\n📋 LICENSE TYPE BREAKDOWN:")
    print(f"  ├─ NLR: {summary['NLR Shipments']:,}")
    print(f"  ├─ LICENSE: {summary['License Shipments']:,}")
    print(f"  └─ EXCEPTION: {summary['Exception Shipments']:,}")
    
    print("\n🏢 BUSINESS UNITS:")
    print(df_final['exporter_business_unit'].value_counts().to_string())
    
    print("\n🌍 TOP DESTINATION COUNTRIES:")
    print(df_final['consignee_country'].value_counts().head(5).to_string())
    
    print("\n📅 SHIPMENT MONTH DISTRIBUTION:")
    print(df_final['shipment_month'].value_counts().sort_index().to_string())
    
    print("\n" + "="*70)
    print("✅ Data generation complete! Ready for analysis.")
    print("="*70)
    
    return df_final, summary

# Execute the data generation
if __name__ == "__main__":
    df, summary = generate_dataset()

📦 EXPORT CONTROLS DATA GENERATOR (WITH DATES)

Generating synthetic export compliance dataset with dates...
  ├─ Creating 5,000 baseline records...
  ├─ Injecting compliance defects...
  ├─ Calculating summary statistics...

✅ DATASET GENERATED SUCCESSFULLY!

📊 DATASET SUMMARY:
  ├─ File: us_export_compliance_data.csv
  ├─ Location: c:\Users\Greg\OneDrive\Desktop\PYTHON PROJECTS\Export_Controls_Project\us_export_compliance_data.csv
  ├─ Total Shipments: 5,000
  ├─ Total Columns: 25
  ├─ Date Range: 2025-07-07 to 2026-07-06
  ├─ Average Shipment Value: $633,792.71
  ├─ Total Value: $3,168,963,549.17
  ├─ Average Clearance: 45.2 hrs

🔴 DEFECTS INJECTED:
  ├─ Address Blindspots: 100
  ├─ FDPR Misclassifications: 16
  ├─ PGA Missing Approvals: 12
  ├─ License Mismatches: 25
  ├─ Restricted Entities: 32
  ├─ EEI Threshold Violations: 2211
  ├─ Port Delays Injected: 411
  └─ Hardcoded Examples: 3 (rows 12, 45, 88)

📋 LICENSE TYPE BREAKDOWN:
  ├─ NLR: 2,218
  ├─ LICENSE: 1,789
  └─ EXCEPTION:

In [5]:
df = pd.read_csv('us_export_compliance_data.csv')
df.describe(include="all")

,shipment_id,shipment_date,clearance_date,exporter_business_unit,consignee_name,consignee_address,schedule_b_code,eccn,bom_origin_country,bom_tooling_origin,...,clearance_hours,pga_required,pga_obtained,trusted_trader_status,total_value_usd,consignee_country,shipment_month,shipment_year,shipment_quarter,shipment_day_of_week
count,5000,5000,5000,5000,5000,5000,5000,5000,5000,5000,...,5000.000000,5000,5000,5000,5.000000e+03,5000,5000.000000,5000.000000,5000.000000,5000.000000
unique,5000,365,3778,5,17,40,9,5,9,6,...,NaN,2,2,2,NaN,12,NaN,NaN,NaN,NaN
top,EXP-100000,2025-09-03,2026-02-20 22:00:00,US-ConsumerElectronics,GLOBAL LOGISTICS LTD,"456 Industrial Zone, Riyadh, SA",9018.90.7500,EAR99,US,US,...,NaN,N,Y,N,NaN,SA,NaN,NaN,NaN,NaN
freq,1,24,5,1582,521,666,606,2265,1448,1817,...,NaN,4018,4478,2967,NaN,666,NaN,NaN,NaN,NaN
mean,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,45.178400,NaN,NaN,NaN,6.337927e+05,NaN,6.639800,2025.501400,2.546600,3.028400
std,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,21.338642,NaN,NaN,NaN,5.532567e+05,NaN,3.454684,0.500048,1.118785,2.013058
min,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,12.000000,NaN,NaN,NaN,1.272700e+02,NaN,1.000000,2025.000000,1.000000,0.000000
25%,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,28.000000,NaN,NaN,NaN,1.757138e+05,NaN,4.000000,2025.000000,2.000000,1.000000
50%,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,44.000000,NaN,NaN,NaN,4.658966e+05,NaN,7.000000,2026.000000,3.000000,3.000000
75%,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,60.000000,NaN,NaN,NaN,9.756829e+05,NaN,10.000000,2026.000000,4.000000,5.000000
